# Classical Model Prediction Pipeline

This notebook uses the trained classical ML models to predict **MaxTemp** and **RainTomorrow**
for a given date, producing the same CSV format as the deep-learning path.

**New:** Supports predictions for a **specific location** or all cities.

Target CSV columns (one row per city):
```
Ville,Date_Predite,Temp_Predite,Prob_Pluie,Pluie_Prevue
```

## Cell 1 — Imports and config

In [29]:
import pandas as pd
import numpy as np
import joblib
from classical_models import FEATURE_COLUMNS, TARGET_RAIN, LOCATION_COLUMN, DATA_PATH

DATE = "2018-03-18"          # change as needed
LOCATION = None               # None = all cities, or e.g. "Sydney"

# Feature columns for rain classification (full set)
RAIN_FEATURES = FEATURE_COLUMNS

# Feature columns for temp regression (MaxTemp excluded to avoid leakage)
TEMP_FEATURES = [c for c in FEATURE_COLUMNS if c != "MaxTemp"]

## Cell 2 — Load models

In [30]:
rain_model = joblib.load("saved_models/xgboost.joblib")        # best classical rain model
temp_model = joblib.load("saved_models/temp_regressor.joblib") # temperature regressor

## Cell 3 — Prediction function

In [31]:
def predict_for_specific_day_classical(
    rain_model, 
    temp_model, 
    df: pd.DataFrame, 
    target_date: str, 
    location: str = None
) -> pd.DataFrame:
    """
    For every city (or a specific city) that has a row on `target_date`, predict:
      - Temp_Predite  : MaxTemp (°C) via regression model
      - Prob_Pluie    : P(RainTomorrow=1) via rain classifier
      - Pluie_Prevue  : 1 if Prob_Pluie > 0.5 else 0

    Parameters
    ----------
    rain_model : fitted rain classifier
    temp_model : fitted temperature regressor
    df : DataFrame with all features + Location column
    target_date : date string e.g. "2017-03-18"
    location : optional, filter to a specific city (e.g. "Sydney")

    Returns a DataFrame with columns:
        Ville, Date_Predite, Temp_Predite, Prob_Pluie, Pluie_Prevue
    """
    day_df = df.copy()

    # 1. Filtre de localisation si demandé
    if location is not None:
        day_df = day_df[day_df[LOCATION_COLUMN] == location].copy()
        if day_df.empty:
            raise ValueError(f"No data found for location '{location}'")

    # 2. IMPORTANT : Sécurité anti-doublons pour éviter le MemoryError
    # On s'assure qu'on ne traite qu'une seule ligne par ville
    day_df = day_df.drop_duplicates(subset=[LOCATION_COLUMN])

    # 3. Prédiction Pluie
    rain_df = day_df.dropna(subset=RAIN_FEATURES)
    X_rain = rain_df[RAIN_FEATURES]
    prob_rain = rain_model.predict_proba(X_rain)[:, 1]

    # 4. Prédiction Température
    temp_df = day_df.dropna(subset=TEMP_FEATURES)
    X_temp = temp_df[TEMP_FEATURES]
    pred_temp = temp_model.predict(X_temp)

    # 5. Création des DataFrames intermédiaires
    rain_result = pd.DataFrame({
        LOCATION_COLUMN: rain_df[LOCATION_COLUMN].values,
        "Prob_Pluie": prob_rain,
    })
    temp_result = pd.DataFrame({
        LOCATION_COLUMN: temp_df[LOCATION_COLUMN].values,
        "Temp_Predite": pred_temp,
    })

    # 6. Le merge (maintenant sécurisé car drop_duplicates a été fait)
    merged = rain_result.merge(temp_result, on=LOCATION_COLUMN, how="inner")

    # 7. Formatage de sortie
    output = pd.DataFrame({
        "Ville":        merged[LOCATION_COLUMN],
        "Date_Predite": target_date,
        "Temp_Predite": merged["Temp_Predite"].round(2),
        "Prob_Pluie":   merged["Prob_Pluie"].round(4),
        "Pluie_Prevue": (merged["Prob_Pluie"] > 0.5).astype(int),
    })

    return output

## Cell 4 — Run prediction (all cities)

In [32]:
import os

df = pd.read_csv(DATA_PATH)
results = predict_for_specific_day_classical(rain_model, temp_model, df, DATE)

os.makedirs("predict", exist_ok=True)
csv_path = f"predict/predict_{DATE}.csv"
results.to_csv(csv_path, index=False)
print(f"Fichier sauvegardé : {csv_path}")

results

Fichier sauvegardé : predict/predict_2018-03-18.csv


,Ville,Date_Predite,Temp_Predite,Prob_Pluie,Pluie_Prevue
0,Adelaide,2018-03-18,16.24,0.8863,1
1,Albany,2018-03-18,18.86,0.4097,0
2,Albury,2018-03-18,23.03,0.0838,0
3,AliceSprings,2018-03-18,38.64,0.7509,1
4,BadgerysCreek,2018-03-18,32.17,0.4659,0
5,Ballarat,2018-03-18,17.15,0.2835,0
6,Bendigo,2018-03-18,20.87,0.0511,0
7,Brisbane,2018-03-18,25.37,0.1658,0
8,Cairns,2018-03-18,32.32,0.6612,1
9,Canberra,2018-03-18,24.95,0.1782,0


## Cell 5 — Run prediction (specific location)

In [33]:
"""# Predict for a specific city
if LOCATION is not None:
    results_loc = predict_for_specific_day_classical(
        rain_model, temp_model, df, DATE, location=LOCATION
    )
    display(results_loc)
else:
    # Demo with Sydney
    print("\n--- Demo: prediction for Sydney ---")
    results_sydney = predict_for_specific_day_classical(
        rain_model, temp_model, df, DATE, location="Sydney"
    )
    display(results_sydney)"""

'# Predict for a specific city\nif LOCATION is not None:\n    results_loc = predict_for_specific_day_classical(\n        rain_model, temp_model, df, DATE, location=LOCATION\n    )\n    display(results_loc)\nelse:\n    # Demo with Sydney\n    print("\n--- Demo: prediction for Sydney ---")\n    results_sydney = predict_for_specific_day_classical(\n        rain_model, temp_model, df, DATE, location="Sydney"\n    )\n    display(results_sydney)'

## Cell 6 — Evaluation against actuals

In [34]:
def evaluate_classical_performance(
    rain_model,
    temp_model,
    df: pd.DataFrame,
    target_date: str,
    location: str = None,
):
    """
    Compare predictions against actuals for target_date.
    Optionally filter by location.
    Returns (mean_absolute_temp_error, rain_accuracy).
    """
    df = df.copy()
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        target_dt = pd.to_datetime(target_date)
        day_df = df[df['Date'] == target_dt].copy()
    else:
        day_df = df.copy()

    # Location filter
    if location is not None:
        day_df = day_df[day_df[LOCATION_COLUMN] == location].copy()

    day_df = day_df.dropna(subset=RAIN_FEATURES + ["MaxTemp", TARGET_RAIN])

    if day_df.empty:
        print(f"No data available for evaluation on {target_date}")
        return None, None

    X_rain = day_df[RAIN_FEATURES]
    X_temp = day_df[TEMP_FEATURES]

    prob_rain   = rain_model.predict_proba(X_rain)[:, 1]
    pred_labels = (prob_rain > 0.5).astype(int)
    pred_temp   = temp_model.predict(X_temp)

    actual_temp = day_df["MaxTemp"].values
    actual_rain = day_df[TARGET_RAIN].values   # already binary (0/1)

    mae_temp    = float(np.mean(np.abs(pred_temp - actual_temp)))
    rain_acc    = float(np.mean(pred_labels == actual_rain))

    loc_label = f" ({location})" if location else ""
    print(f"--- Performances au {target_date}{loc_label} (classical models) ---")
    print(f"Écart de température moyen : {mae_temp:.2f}°C")
    print(f"Accuracy Pluie             : {rain_acc * 100:.1f}%")
    return mae_temp, rain_acc


# Evaluate all cities
mae, acc = evaluate_classical_performance(rain_model, temp_model, df, DATE)

# Evaluate for specific location if set
if LOCATION is not None:
    mae_loc, acc_loc = evaluate_classical_performance(
        rain_model, temp_model, df, DATE, location=LOCATION
    )

--- Performances au 2018-03-18 (classical models) ---
Écart de température moyen : 0.45°C
Accuracy Pluie             : 82.4%


## Sanity Checks

| Check | Expected |
|-------|----------|
| `results.columns.tolist()` | `['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']` |
| `results["Temp_Predite"].between(5, 50).all()` | `True` (no negative temps) |
| `results["Prob_Pluie"].between(0, 1).all()` | `True` |
| `results["Pluie_Prevue"].isin([0, 1]).all()` | `True` |
| CSV file exists at `predict/predict_{DATE}.csv` | `True` |

In [35]:
print("Columns:", results.columns.tolist())
print("Expected: ['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']")
print()
print(f"Temp in [5, 50]: {results['Temp_Predite'].between(5, 50).all()}")
print(f"Prob in [0, 1]:  {results['Prob_Pluie'].between(0, 1).all()}")
print(f"Rain in {{0,1}}: {results['Pluie_Prevue'].isin([0, 1]).all()}")
# print(f"CSV exists:      {os.path.exists(f'predict/predict_{DATE}.csv')}")

Columns: ['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']
Expected: ['Ville', 'Date_Predite', 'Temp_Predite', 'Prob_Pluie', 'Pluie_Prevue']

Temp in [5, 50]: True
Prob in [0, 1]:  True
Rain in {0,1}: True
